# Chebyshev Iteration Method — Visual Guide (Go)

This notebook provides interactive visualizations of the [Chebyshev semi-iterative method](https://en.wikipedia.org/wiki/Chebyshev_iteration) for solving **Ax = b**, where **A** is symmetric positive-definite.

All code uses the [chebyshev-iteration](https://github.com/peterborodatyy/chebyshev-iteration) Go library directly.

**Contents:**
1. Helpers & imports
2. Convergence: Chebyshev vs Richardson iteration
3. Parameter ordering comparison (Direct / Reverse / Alternating)
4. Effect of eigenvalue bound tightness
5. Chebyshev polynomial roots on the eigenvalue spectrum
6. Error polynomial
7. Convergence rate vs condition number

**Requirements:** [gophernotes](https://github.com/gopherdata/gophernotes) Go kernel for Jupyter.

## 1. Imports & Helpers

We import `gonum/plot` for charting and define helper functions used throughout.

In [ ]:
import (
	"fmt"
	"image/color"
	"math"

	"github.com/peterborodatyy/chebyshev-iteration/chebyshev"
	"gonum.org/v1/plot"
	"gonum.org/v1/plot/plotter"
	"gonum.org/v1/plot/vg"
)

In [ ]:
// buildTridiag builds an n×n tridiagonal SPD matrix with given diagonal and off-diagonal 1.
func buildTridiag(n int, diag float64) [][]float64 {
	A := make([][]float64, n)
	for i := range A {
		A[i] = make([]float64, n)
		A[i][i] = diag
		if i > 0 {
			A[i][i-1] = 1.0
		}
		if i < n-1 {
			A[i][i+1] = 1.0
		}
	}
	return A
}

// eigenBoundsTridiag returns exact eigenvalue bounds for tridiagonal(diag, 1).
func eigenBoundsTridiag(n int, diag float64) (lmin, lmax float64) {
	lmin = diag + 2*math.Cos(float64(n)*math.Pi/float64(n+1))
	lmax = diag + 2*math.Cos(math.Pi/float64(n+1))
	return
}

// residualVec computes b - A*x.
func residualVec(A [][]float64, b, x []float64) []float64 {
	n := len(b)
	r := make([]float64, n)
	for i := 0; i < n; i++ {
		sum := 0.0
		for j := 0; j < n; j++ {
			sum += A[i][j] * x[j]
		}
		r[i] = b[i] - sum
	}
	return r
}

// vecNorm returns the L2 norm of v.
func vecNorm(v []float64) float64 {
	sum := 0.0
	for _, x := range v {
		sum += x * x
	}
	return math.Sqrt(sum)
}

// tau computes the Chebyshev relaxation parameter for step k out of n total.
func tau(k, n int, lmin, lmax float64) float64 {
	cosArg := math.Pi * (2*float64(k) - 1) / (2 * float64(n))
	return 2.0 / (lmin + lmax + (lmax-lmin)*math.Cos(cosArg))
}

// orderingSeq generates parameter index sequence for a given ordering.
func orderingSeq(ord chebyshev.Ordering, n int) []int {
	seq := make([]int, n)
	switch ord {
	case chebyshev.Direct:
		for i := range seq { seq[i] = i + 1 }
	case chebyshev.Reverse:
		for i := range seq { seq[i] = n - i }
	case chebyshev.Alternating:
		lo, hi := 1, n
		for i := range seq {
			if i%2 == 0 { seq[i] = lo; lo++ } else { seq[i] = hi; hi-- }
		}
	}
	return seq
}

// chebyshevTrace solves Ax=b with Chebyshev iteration and records per-iteration residuals.
func chebyshevTrace(A [][]float64, b []float64, lmin, lmax float64, maxIter int, tol float64, ord chebyshev.Ordering) []float64 {
	n := len(b)
	x := make([]float64, n)
	seq := orderingSeq(ord, maxIter)
	var residuals []float64
	for _, k := range seq {
		r := residualVec(A, b, x)
		rn := vecNorm(r)
		residuals = append(residuals, rn)
		if rn <= tol || math.IsNaN(rn) || math.IsInf(rn, 0) { break }
		tk := tau(k, maxIter, lmin, lmax)
		for i := range x { x[i] += tk * r[i] }
	}
	return residuals
}

// richardsonTrace solves Ax=b with fixed Richardson iteration and records per-iteration residuals.
func richardsonTrace(A [][]float64, b []float64, lmin, lmax float64, maxIter int, tol float64) []float64 {
	n := len(b)
	x := make([]float64, n)
	tauFixed := 2.0 / (lmin + lmax)
	var residuals []float64
	for iter := 0; iter < maxIter; iter++ {
		r := residualVec(A, b, x)
		rn := vecNorm(r)
		residuals = append(residuals, rn)
		if rn <= tol || math.IsNaN(rn) || math.IsInf(rn, 0) { break }
		for i := range x { x[i] += tauFixed * r[i] }
	}
	return residuals
}

// resToXYs converts a residual slice to plotter.XYs (iteration, log10(residual)).
func resToXYs(residuals []float64) plotter.XYs {
	pts := make(plotter.XYs, len(residuals))
	for i, r := range residuals {
		pts[i].X = float64(i + 1)
		pts[i].Y = r
	}
	return pts
}

// savePlot saves a plot to PNG.
func savePlot(p *plot.Plot, filename string) {
	if err := p.Save(25*vg.Centimeter, 15*vg.Centimeter, filename); err != nil {
		fmt.Printf("Error saving %s: %v\n", filename, err)
	} else {
		fmt.Printf("Saved: %s\n", filename)
	}
}

fmt.Println("Helpers loaded.")

## 2. Chebyshev vs Richardson Iteration

The key advantage of Chebyshev iteration over simple (fixed-parameter) Richardson iteration is the use of **optimal varying parameters** derived from Chebyshev polynomial roots. This dramatically accelerates convergence.

We compare both methods on a 20×20 tridiagonal SPD system.

The Chebyshev iteration solves **Ax = b** via:

$$x_{k+1} = x_k + \tau_k \, (b - A x_k)$$

where the relaxation parameters $\tau_k$ are reciprocals of the shifted Chebyshev roots:

$$\tau_k = \frac{2}{\lambda_{\min} + \lambda_{\max} + (\lambda_{\max} - \lambda_{\min}) \cos\!\left(\frac{\pi(2k-1)}{2n}\right)}$$

In [ ]:
{
	n := 20
	A := buildTridiag(n, 4.0)
	b := make([]float64, n)
	for i := range b { b[i] = 1.0 }
	lmin, lmax := eigenBoundsTridiag(n, 4.0)
	fmt.Printf("Eigenvalue range: [%.4f, %.4f], condition number: %.1f\n", lmin, lmax, lmax/lmin)

	resCheb := chebyshevTrace(A, b, lmin, lmax, 200, 1e-14, chebyshev.Direct)
	resRich := richardsonTrace(A, b, lmin, lmax, 200, 1e-14)

	p := plot.New()
	p.Title.Text = "Chebyshev vs Richardson Iteration"
	p.X.Label.Text = "Iteration"
	p.Y.Label.Text = "Residual ‖b − Ax‖₂"
	p.Y.Scale = plot.LogScale{}
	p.Y.Tick.Marker = plot.LogTicks{Prec: -1}

	lCheb, _ := plotter.NewLine(resToXYs(resCheb))
	lCheb.Color = color.RGBA{R: 33, G: 150, B: 243, A: 255}
	lCheb.Width = vg.Points(2)

	lRich, _ := plotter.NewLine(resToXYs(resRich))
	lRich.Color = color.RGBA{R: 244, G: 67, B: 54, A: 255}
	lRich.Width = vg.Points(2)
	lRich.Dashes = []vg.Length{vg.Points(5), vg.Points(3)}

	p.Add(lCheb, lRich, plotter.NewGrid())
	p.Legend.Add("Chebyshev iteration", lCheb)
	p.Legend.Add("Richardson iteration (fixed τ)", lRich)
	p.Legend.Top = true

	savePlot(p, "01_chebyshev_vs_richardson.png")

	// Also verify using the library's Solver
	s := chebyshev.Solver{A: A, B: b, LambdaMin: lmin, LambdaMax: lmax, MaxIter: 200, Tolerance: 1e-14}
	result, _ := s.Solve()
	fmt.Printf("Library Solver: converged=%v, iterations=%d, residual=%.4e\n", result.Converged, result.Iterations, result.Residual)
}

## 3. Parameter Ordering Comparison

The Chebyshev parameters can be applied in different orders. In exact arithmetic, all orderings produce the same result after a full cycle. In floating-point arithmetic, the ordering affects numerical stability.

- **Direct:** k = 1, 2, ..., n
- **Reverse:** k = n, n−1, ..., 1
- **Alternating:** k = 1, n, 2, n−1, ... (interleaves small and large parameters)

In [ ]:
{
	n := 30
	A := buildTridiag(n, 4.0)
	b := make([]float64, n)
	for i := range b { b[i] = math.Sin(float64(i+1) * 0.7) }
	lmin, lmax := eigenBoundsTridiag(n, 4.0)

	type ordConfig struct {
		name  string
		ord   chebyshev.Ordering
		color color.RGBA
	}
	orderings := []ordConfig{
		{"Direct", chebyshev.Direct, color.RGBA{R: 33, G: 150, B: 243, A: 255}},
		{"Reverse", chebyshev.Reverse, color.RGBA{R: 255, G: 152, B: 0, A: 255}},
		{"Alternating", chebyshev.Alternating, color.RGBA{R: 76, G: 175, B: 80, A: 255}},
	}

	p := plot.New()
	p.Title.Text = "Convergence by Parameter Ordering"
	p.X.Label.Text = "Iteration"
	p.Y.Label.Text = "Residual ‖b − Ax‖₂"
	p.Y.Scale = plot.LogScale{}
	p.Y.Tick.Marker = plot.LogTicks{Prec: -1}
	p.Add(plotter.NewGrid())

	for _, o := range orderings {
		res := chebyshevTrace(A, b, lmin, lmax, 512, 1e-14, o.ord)
		l, _ := plotter.NewLine(resToXYs(res))
		l.Color = o.color
		l.Width = vg.Points(2)
		p.Add(l)
		p.Legend.Add(fmt.Sprintf("%s (%d iter)", o.name, len(res)), l)

		// Also run via library Solver
		s := chebyshev.Solver{A: A, B: b, LambdaMin: lmin, LambdaMax: lmax, MaxIter: 512, Tolerance: 1e-14}
		result, _ := s.SolveWithOrdering(o.ord)
		fmt.Printf("%-12s  iterations=%d  residual=%.4e\n", o.name, result.Iterations, result.Residual)
	}
	p.Legend.Top = true

	savePlot(p, "02_parameter_ordering.png")
}

## 4. Effect of Eigenvalue Bound Tightness

The convergence rate depends on the ratio $\kappa = \lambda_{\max}/\lambda_{\min}$. Tighter bounds (closer to the true extreme eigenvalues) mean faster convergence. Here we show what happens when bounds are exact, 2× loose, and 5× loose.

In [ ]:
{
	n := 20
	A := buildTridiag(n, 4.0)
	b := make([]float64, n)
	for i := range b { b[i] = 1.0 }
	lminExact, lmaxExact := eigenBoundsTridiag(n, 4.0)

	type boundConfig struct {
		label string
		lmin  float64
		lmax  float64
		color color.RGBA
	}
	configs := []boundConfig{
		{"Exact bounds", lminExact, lmaxExact, color.RGBA{R: 76, G: 175, B: 80, A: 255}},
		{"2× looser", lminExact / 2, lmaxExact * 2, color.RGBA{R: 255, G: 152, B: 0, A: 255}},
		{"5× looser", lminExact / 5, lmaxExact * 5, color.RGBA{R: 244, G: 67, B: 54, A: 255}},
	}

	p := plot.New()
	p.Title.Text = "Effect of Eigenvalue Bound Tightness"
	p.X.Label.Text = "Iteration"
	p.Y.Label.Text = "Residual ‖b − Ax‖₂"
	p.Y.Scale = plot.LogScale{}
	p.Y.Tick.Marker = plot.LogTicks{Prec: -1}
	p.Add(plotter.NewGrid())

	for _, c := range configs {
		kappa := c.lmax / c.lmin
		res := chebyshevTrace(A, b, c.lmin, c.lmax, 300, 1e-14, chebyshev.Alternating)
		l, _ := plotter.NewLine(resToXYs(res))
		l.Color = c.color
		l.Width = vg.Points(2)
		p.Add(l)
		p.Legend.Add(fmt.Sprintf("%s (κ=%.0f, %d iter)", c.label, kappa, len(res)), l)
		fmt.Printf("%s: κ=%.0f, %d iterations\n", c.label, kappa, len(res))
	}
	p.Legend.Top = true

	savePlot(p, "03_eigenvalue_bounds.png")
}

## 5. Chebyshev Polynomial Roots on the Eigenvalue Spectrum

The Chebyshev parameters $\tau_k = 1/d_k$ where $d_k$ are the roots of the Chebyshev polynomial $T_n$ shifted to $[\lambda_{\min}, \lambda_{\max}]$. These roots cluster near the endpoints, which is precisely what makes the method effective — the error polynomial is minimized where eigenvalues are most likely to concentrate.

In [ ]:
{
	nPoly := 32
	nMat := 20
	A := buildTridiag(nMat, 4.0)
	_ = A // used for eigenvalue computation
	lmin, lmax := eigenBoundsTridiag(nMat, 4.0)

	// Compute eigenvalues of the tridiagonal matrix
	eigvals := make([]float64, nMat)
	for i := 0; i < nMat; i++ {
		eigvals[i] = 4.0 + 2*math.Cos(float64(i+1)*math.Pi/float64(nMat+1))
	}

	// Shifted Chebyshev roots
	roots := make([]float64, nPoly)
	for k := 0; k < nPoly; k++ {
		roots[k] = (lmin+lmax)/2 + (lmax-lmin)/2*math.Cos(math.Pi*(2*float64(k+1)-1)/(2*float64(nPoly)))
	}

	// Plot: eigenvalues and Chebyshev roots on a scatter plot
	p := plot.New()
	p.Title.Text = fmt.Sprintf("Chebyshev Roots (n=%d) vs Eigenvalues on [λ_min, λ_max]", nPoly)
	p.X.Label.Text = "Position on [λ_min, λ_max]"
	p.Y.Label.Text = ""

	// Eigenvalues at y=1
	eigPts := make(plotter.XYs, nMat)
	for i, ev := range eigvals {
		eigPts[i].X = ev
		eigPts[i].Y = 1.0
	}
	sEig, _ := plotter.NewScatter(eigPts)
	sEig.GlyphStyle.Color = color.RGBA{R: 244, G: 67, B: 54, A: 255}
	sEig.GlyphStyle.Radius = vg.Points(4)

	// Chebyshev roots at y=0
	rootPts := make(plotter.XYs, nPoly)
	for i, r := range roots {
		rootPts[i].X = r
		rootPts[i].Y = 0.0
	}
	sRoot, _ := plotter.NewScatter(rootPts)
	sRoot.GlyphStyle.Color = color.RGBA{R: 33, G: 150, B: 243, A: 255}
	sRoot.GlyphStyle.Radius = vg.Points(3)

	p.Add(sEig, sRoot)
	p.Legend.Add("Eigenvalues of A", sEig)
	p.Legend.Add(fmt.Sprintf("Chebyshev roots (n=%d)", nPoly), sRoot)
	p.Legend.Top = true
	p.Y.Min = -0.5
	p.Y.Max = 1.5

	savePlot(p, "04_chebyshev_roots.png")

	// Histogram of root distribution
	ph := plot.New()
	ph.Title.Text = "Distribution of Chebyshev Roots"
	ph.X.Label.Text = "Position on [λ_min, λ_max]"
	ph.Y.Label.Text = "Root count"

	rootVals := make(plotter.Values, nPoly)
	for i, r := range roots { rootVals[i] = r }
	hist, _ := plotter.NewHist(rootVals, 20)
	hist.FillColor = color.RGBA{R: 33, G: 150, B: 243, A: 180}
	ph.Add(hist)

	savePlot(ph, "05_root_distribution.png")
}

## 6. The Error Polynomial

The Chebyshev iteration minimizes the maximum of the error polynomial $p_n(\lambda)$ over $[\lambda_{\min}, \lambda_{\max}]$, where $p_n(\lambda) = \prod_{k=1}^{n}(1 - \tau_k \lambda)$. This is exactly the [minimax property](https://en.wikipedia.org/wiki/Chebyshev_polynomials#Properties) of Chebyshev polynomials — no other degree-n polynomial with $p_n(0) = 1$ has a smaller maximum on the interval.

In [ ]:
{
	lmin, lmax := 2.0, 6.0
	nLambda := 500
	lambdas := make([]float64, nLambda)
	for i := range lambdas {
		lambdas[i] = lmin + (lmax-lmin)*float64(i)/float64(nLambda-1)
	}

	for _, nPoly := range []int{8, 32} {
		p := plot.New()
		p.Title.Text = fmt.Sprintf("Error Polynomial |p_%d(λ)|  on [%.0f, %.0f]", nPoly, lmin, lmax)
		p.X.Label.Text = "λ"
		p.Y.Label.Text = "|p_n(λ)|"
		p.Y.Scale = plot.LogScale{}
		p.Y.Tick.Marker = plot.LogTicks{Prec: -1}
		p.Add(plotter.NewGrid())

		// Chebyshev error polynomial
		chebPts := make(plotter.XYs, nLambda)
		for i, lam := range lambdas {
			prod := 1.0
			for k := 1; k <= nPoly; k++ {
				tk := tau(k, nPoly, lmin, lmax)
				prod *= (1 - tk*lam)
			}
			chebPts[i].X = lam
			chebPts[i].Y = math.Abs(prod)
		}
		lCheb, _ := plotter.NewLine(chebPts)
		lCheb.Color = color.RGBA{R: 33, G: 150, B: 243, A: 255}
		lCheb.Width = vg.Points(2)

		// Richardson error polynomial
		richPts := make(plotter.XYs, nLambda)
		tauRich := 2.0 / (lmin + lmax)
		for i, lam := range lambdas {
			richPts[i].X = lam
			richPts[i].Y = math.Abs(math.Pow(1-tauRich*lam, float64(nPoly)))
		}
		lRich, _ := plotter.NewLine(richPts)
		lRich.Color = color.RGBA{R: 244, G: 67, B: 54, A: 255}
		lRich.Width = vg.Points(2)
		lRich.Dashes = []vg.Length{vg.Points(5), vg.Points(3)}

		p.Add(lCheb, lRich)
		p.Legend.Add("Chebyshev (optimal)", lCheb)
		p.Legend.Add("Richardson (fixed τ)", lRich)
		p.Legend.Top = true

		maxCheb := 0.0
		for _, pt := range chebPts { if pt.Y > maxCheb { maxCheb = pt.Y } }
		maxRich := 0.0
		for _, pt := range richPts { if pt.Y > maxRich { maxRich = pt.Y } }
		fmt.Printf("n=%d: max|p| Chebyshev=%.2e, Richardson=%.2e\n", nPoly, maxCheb, maxRich)

		savePlot(p, fmt.Sprintf("06_error_polynomial_n%d.png", nPoly))
	}
}

## 7. Convergence Rate vs Condition Number

The theoretical convergence factor per iteration is $\rho = \frac{1 - \sqrt{\xi}}{1 + \sqrt{\xi}}$ where $\xi = \lambda_{\min}/\lambda_{\max}$. As the condition number grows, convergence slows dramatically.

In [ ]:
{
	// Theoretical convergence factor
	nPts := 200
	rhoPts := make(plotter.XYs, nPts)
	iterPts := make(plotter.XYs, nPts)
	for i := 0; i < nPts; i++ {
		kappa := 1.5 + 198.5*float64(i)/float64(nPts-1)
		xi := 1.0 / kappa
		rho := (1 - math.Sqrt(xi)) / (1 + math.Sqrt(xi))
		iters := math.Log(1e-10) / math.Log(rho)
		rhoPts[i].X = kappa
		rhoPts[i].Y = rho
		iterPts[i].X = kappa
		iterPts[i].Y = iters
	}

	// Plot 1: Convergence factor
	p1 := plot.New()
	p1.Title.Text = "Convergence Factor vs Condition Number"
	p1.X.Label.Text = "Condition number κ = λ_max / λ_min"
	p1.Y.Label.Text = "Convergence factor ρ"
	p1.Add(plotter.NewGrid())

	lRho, _ := plotter.NewLine(rhoPts)
	lRho.Color = color.RGBA{R: 33, G: 150, B: 243, A: 255}
	lRho.Width = vg.Points(2)
	p1.Add(lRho)

	savePlot(p1, "07_convergence_factor.png")

	// Plot 2: Iterations needed
	p2 := plot.New()
	p2.Title.Text = "Iterations Needed vs Condition Number"
	p2.X.Label.Text = "Condition number κ"
	p2.Y.Label.Text = "Iterations for 1e-10 reduction"
	p2.Add(plotter.NewGrid())

	lIter, _ := plotter.NewLine(iterPts)
	lIter.Color = color.RGBA{R: 33, G: 150, B: 243, A: 255}
	lIter.Width = vg.Points(2)
	p2.Add(lIter)

	savePlot(p2, "08_iterations_needed.png")

	// Print notable points
	fmt.Println("\nNotable points:")
	for _, kappa := range []float64{5, 20, 50, 100} {
		xi := 1.0 / kappa
		rho := (1 - math.Sqrt(xi)) / (1 + math.Sqrt(xi))
		iters := math.Log(1e-10) / math.Log(rho)
		fmt.Printf("  κ=%3.0f: ρ=%.4f, iterations=%.0f\n", kappa, rho, iters)
	}

	// Verify with actual solves
	fmt.Println("\nVerification with actual solves (20×20, alternating):")
	for _, diag := range []float64{4.0, 6.0, 12.0, 22.0} {
		n := 20
		A := buildTridiag(n, diag)
		b := make([]float64, n)
		for i := range b { b[i] = 1.0 }
		lmin, lmax := eigenBoundsTridiag(n, diag)
		kappa := lmax / lmin

		s := chebyshev.Solver{A: A, B: b, LambdaMin: lmin, LambdaMax: lmax, MaxIter: 2000, Tolerance: 1e-10}
		result, _ := s.SolveWithOrdering(chebyshev.Alternating)
		fmt.Printf("  κ=%5.1f: actual=%d iterations, residual=%.4e\n", kappa, result.Iterations, result.Residual)
	}
}

## Summary

| Visualization | Key Insight |
|---|---|
| Chebyshev vs Richardson | Chebyshev parameters accelerate convergence by orders of magnitude |
| Ordering comparison | Alternating ordering is most numerically stable |
| Eigenvalue bound tightness | Loose bounds still converge, but much slower |
| Chebyshev roots | Roots cluster at endpoints of the eigenvalue interval |
| Error polynomial | Chebyshev achieves the minimax-optimal error polynomial |
| Condition number | Iterations scale as $O(\sqrt{\kappa})$, practical for $\kappa < 100$ |

All examples use the [chebyshev-iteration](https://github.com/peterborodatyy/chebyshev-iteration) Go library.